# 07 - Generator lineages: whose stories make the best emotion probes?

**What this notebook is for.** An emotion probe here is a direction in the
activations of one language model, built by having that model read short
stories about an emotion and averaging what its residual stream does. The
stories have to come from somewhere, and somebody has to write them. This
notebook asks whether it matters *who*. Two experiments answer it: the
third-lineage test (registry name: experiment E11), which separates "the
writer must be the probed model itself" from "the writer must simply be
good", and the scale-and-diversity dose-response (registry name: E12), which
asks how many stories are enough and whether deliberately varying the writing
prompt helps.

**Model probed throughout:
[`google/gemma-4-31b-it`](https://huggingface.co/google/gemma-4-31b-it)**, the
instruction-tuned model. The generators only write the stories. Every vector
below is extracted from Gemma reading them, so a row labelled "DeepSeek" means
*probes Gemma built while reading DeepSeek-written stories*, never *probes
taken out of DeepSeek*.

**Where the question came from.** Two earlier results left an ambiguity.
Experiments E5 and E6 showed that emotion probes are corpus-dependent objects:
swap the stories and the probe changes. Experiment E10 then showed that
stories the probed model wrote itself beat an external corpus written by a
much smaller model,
[gemma-4-4B](https://huggingface.co/datasets/snae/emotion_stories_gemma_4_4B).
That leaves two explanations tangled together. Does the probed model need its
OWN stories (a distribution match), or does it just need GOOD stories
(generator quality)? E11 untangles them by adding a third corpus from a strong
*external* generator,
[deepseek-v4-pro](https://openrouter.ai/deepseek/deepseek-v4-pro) via
OpenRouter. E12 then varies the size and the prompt diversity of that corpus.

<h3>Key concepts</h3>

<ul>
<li><b>Battery</b>: our fixed set of 12 emotions (happy, inspired, loving,
proud, calm, desperate, angry, guilty, sad, afraid, nervous, surprised). Two
independent sets of 12 test scenarios exist for it: the <i>paper battery</i>
(scenarios from the source paper) and a <i>held-out battery</i> we wrote
ourselves.</li>
<li><b>Contrast probe</b>: one emotion's mean story activation minus the mean
over all 12 emotions, then scaled to unit length. Subtracting the pool mean is
what removes "this is a short story" from the direction and leaves the emotion.</li>
<li><b>Centered cosine</b>: the cosine between a scenario's activation and a
probe, taken after subtracting the mean activation over the whole scenario set
(the readout introduced in experiment E9).</li>
<li><b>Dual-battery bar</b>: the registered pass criterion. A layer passes if
the probes rank the target emotion in the top 3 of 12 for at least 8 of the 12
scenarios, on BOTH batteries. Chance is 3 of 12.</li>
<li><b>RSA</b> (representational similarity analysis): correlate the 12x12
emotion-to-emotion similarity matrices of two probe sets. It asks whether the
<i>shape</i> of emotion space agrees, even when individual directions do not.</li>
<li><b>Elo</b>: a rating fitted from pairwise comparisons, here of how much the
model says it would prefer each of 64 activities.</li>
<li><b>Registry codes</b>: E-numbers (E5, E10, ...) name experiment nodes and
C-numbers name claim nodes in <code>TREE.md</code>; R1 to R4 name an
experiment's analysis reads, each registered before any scoring ran.</li>
</ul>

<h3>Index</h3>

<ol>
<li>Which stories are we comparing? (the three corpora, side by side)</li>
<li>Is the winning corpus just repeating itself? (phrase repetition, read R4)</li>
<li>Whose stories give probes that detect at the most layers? (E11's read R1)</li>
<li>How many stories do probes need, and does prompt diversity help? (E12)</li>
<li>Do the probe directions themselves converge? (E11's read R2)</li>
<li>Where does corpus quality still pay? (the preference read, R3)</li>
<li>What we are saying, exactly (verdict, caveats, and the numbers behind them)</li>
</ol>

<h3>Data lineage</h3>

Every number below is sliced from a committed evidence file by
<code>emotion_vectors.lineage_report</code> and printed beside the figure that
uses it. Nothing in this notebook re-scores anything. The evidence files are
<code>results/e11_lineage.json</code>,
<code>results/e12_scale_curve.json</code>,
<code>results/e12_diverse_fullcorpus_grid.json</code>,
<code>results/e12_pref_matched_n_exploratory.json</code> and
<code>results/raw_vs_chat_extraction_cosine.json</code>, all published in the
[experiment-artifacts dataset](https://huggingface.co/datasets/abotresol/emotion-vectors-experiment-artifacts).
The probe vector bundles, all post-fix per E11's registered convention, are
[self-generated](https://huggingface.co/datasets/abotresol/emotion-selfstory-vectors-gemma-4-31b-it-postfix),
[weak external](https://huggingface.co/datasets/abotresol/emotion-vectors-gemma-4-31b-it-postfix)
(the battery-12 subset),
[fixed DeepSeek](https://huggingface.co/datasets/abotresol/emotion-deepseek-vectors-gemma-4-31b-it)
and
[diverse DeepSeek](https://huggingface.co/datasets/abotresol/emotion-deepseek-diverse-vectors-gemma-4-31b-it).
The story corpora are linked in the section 1 table. Everything resolves
through <code>emotion_vectors.artifacts.fetch</code>, which looks in the local
<code>results/</code> tree first and downloads from Hugging Face otherwise, so
this notebook runs unchanged on any clone. <code>DATA.md</code> holds the full
index.

In [1]:
# this cell loads the frozen evidence files and the three story corpora; every
# path resolves through emotion_vectors.artifacts.fetch (local results/ first,
# published Hugging Face datasets otherwise)
from emotion_vectors.lineage_report import PASS_BAR, PROBED_MODEL, load_corpora, load_evidence

# LOAD-BEARING VERIFICATION (keep this call as it is): two probe sets were
# scored into BOTH e11_lineage.json and e12_diverse_fullcorpus_grid.json.
# load_evidence refuses to return unless the duplicated rows agree bit for bit
# on the detection read and on the geometry read, so no two figures below can
# quote different numbers for the same measured quantity.
evidence = load_evidence()
corpora = load_corpora()
print(f"probed model: {PROBED_MODEL}; registered pass bar: {PASS_BAR} of 12 scenarios correct")
print(f"evidence files loaded and cross-checked; corpora loaded: {list(corpora)}")

probed model: google/gemma-4-31b-it; registered pass bar: 8 of 12 scenarios correct
evidence files loaded and cross-checked; corpora loaded: ['self-generated (E10)', 'fixed DeepSeek (E11)', 'diverse DeepSeek (E12)']


## 1. Which stories are we comparing?

Three corpora share one instruction core: *write a short third-person story,
around 150 words, about a person experiencing X, without naming X*. A fourth
lineage appears in every later figure but has no corpus loaded here: the **weak
external** corpus written by gemma-4-4B, carried over from experiments E6 and
E10 as the low-quality comparison arm.

<table>
<tr><th>lineage</th><th>who wrote the stories</th><th>stories requested per emotion</th><th>prompt recipe</th><th>story corpus</th></tr>
<tr><td>self-generated</td><td>gemma-4-31b-it, the probed model itself</td><td>256</td><td>the fixed instruction</td><td><a href="https://huggingface.co/datasets/abotresol/emotion-stories-gemma-4-31b-it">emotion-stories-gemma-4-31b-it</a></td></tr>
<tr><td>fixed DeepSeek</td><td>deepseek-v4-pro</td><td>256</td><td>the identical fixed instruction</td><td><a href="https://huggingface.co/datasets/abotresol/emotion-stories-deepseek-v4-pro">emotion-stories-deepseek-v4-pro</a></td></tr>
<tr><td>diverse DeepSeek</td><td>deepseek-v4-pro</td><td>1024</td><td>the instruction plus a persona and a setting pinned from a deterministic 8x8 grid</td><td><a href="https://huggingface.co/datasets/abotresol/emotion-stories-deepseek-v4-pro-diverse">emotion-stories-deepseek-v4-pro-diverse</a></td></tr>
<tr><td>weak external</td><td>gemma-4-4B</td><td>not loaded here</td><td>the open replication's own recipe</td><td><a href="https://huggingface.co/datasets/snae/emotion_stories_gemma_4_4B">snae/emotion_stories_gemma_4_4B</a></td></tr>
</table>

The requested counts and the grid size in that table are the generation
recipe's parameters (`scripts/generate_self_stories.py` and
`scripts/generate_openrouter_stories.py`, the latter with `--diverse`). The
cell below prints what actually survived the leakage and terminal-ending
filters, which is what every later figure uses.

In [2]:
# section 1 is load-call-show: sizes requested against sizes kept, one unpicked
# "sad" story from each corpus, and the self-generated corpus's house habit
from emotion_vectors.lineage_report import corpus_overview

overview = corpus_overview(corpora)
print("\n".join(overview["lines"]))

stories per corpus (requested by the recipe -> kept after filtering):
  self-generated (E10): 3072 requested (256 x 12 emotions) -> 3072 kept; smallest per-emotion count 256
  fixed DeepSeek (E11): 3072 requested (256 x 12 emotions) -> 3070 kept; smallest per-emotion count 255
  diverse DeepSeek (E12): 12288 requested (1024 x 12 emotions) -> 12262 kept; smallest per-emotion count 1018

=== self-generated (E10) | assigned emotion: sad (first story, unpicked) ===
Elias sat on the edge of the unmade bed, staring at the indentation in the pillow beside him. The house was too quiet, the kind of silence that felt heavy, pressing against his chest until breathing became a conscious effort. He reached for a coffee mug, but his hand trembled, and he let it go, watching the steam vanish into the cold morning air.

He didn’t turn on the lights. He preferred the gray dimness that matched the hollow ache behind his ribs. A single tear tracked a slow, salty path dow

=== fixed DeepSeek (E11) | assig

<details><summary><b>How to read this printout</b></summary>

The first block is a reconciliation. "Requested" is what the generation script
asked the writer for; "kept" is what survived the filters that drop a story if
it names the emotion word or ends mid-sentence. The smallest per-emotion count
matters later: the dose-response curves in section 4 can only subsample up to
the emotion with the fewest stories, which is why their largest points are not
the round requested numbers.

The three stories are all answers to the same assignment ("sad"), and each is
the first story of that emotion in its file, not a cherry-picked one. Read them
for texture, not for evidence. The self-generated corpus has a house style: the
same protagonist name recurs, and the last line of the printout counts how
often. A valid reading is "the self-generated corpus is stylistically narrow".
An invalid reading is "the self-generated corpus is therefore bad", because
section 3 shows its probes work; section 2 is where that tension gets measured
instead of eyeballed.

</details>

**What this establishes.** The three corpora are comparable in size and share
an instruction core, so a later difference between them is attributable to the
writer and the prompt recipe rather than to how much data each got. It also
flags the covariate that section 2 has to rule out: the self-generated corpus
looks repetitive.

**Live hypothesis (H-repetition).** The self-generated corpus is degenerate,
collapsed onto one narrative scaffold, and any probe advantage it has comes
from that low variance rather than from matching the probed model's
distribution. **Deciding read:** the phrase-overlap measure in section 2,
combined with the detection result in section 3.

**Open question.** The weak external corpus is not loaded or sampled here, so
its texture is not on the page. Anyone wanting to eyeball it should open the
linked dataset directly.

## 2. Is the winning corpus just repeating itself?

This is the exploratory phrase-repetition read (registry name: read R4 of
E11), added by the extraction-audit session and deliberately fenced off from
the three registered reads. It measures, within each corpus, how often two
different stories share the same run of five consecutive words. High overlap
means the writer keeps reaching for the same phrasing.

The point of measuring it is to separate two explanations that would otherwise
be confounded: "this corpus matches the probed model's distribution" and "this
corpus collapsed onto a single scaffold".

In [3]:
# section 2 is load-call-show: within-corpus 5-gram phrase overlap per lineage,
# with the measured floor drawn as the grading anchor
from emotion_vectors.lineage_report import diversity_figure

fig_diversity, diversity_stats = diversity_figure(evidence)
fig_diversity.show()
print("\n".join(diversity_stats["lines"]))

self-generated overlap 0.0386 is 53x fixed DeepSeek's 0.0007; weak external not measured


<details><summary><b>How to read this figure</b></summary>

The horizontal axis names the corpus and, in parentheses, who wrote it. The
vertical axis is the mean pairwise 5-gram Jaccard overlap: for every pair of
stories in a corpus, the fraction of five-word runs they share, averaged over
pairs. Zero (the bottom axis) means no two stories in the corpus share any
five-word phrase at all, which is maximal diversity. The dotted line marks the
lowest value actually measured here, which is the fixed-DeepSeek corpus, and
the right-hand labels name both anchors.

A good result for the "these corpora are not degenerate" question would be all
bars near the floor. A bad result would be a bar many times the floor. The
observed pattern is the bad-looking one for exactly one corpus, and the ratio
is in the figure title and the printout above.

Valid reading: the self-generated corpus reuses phrasing far more than the
DeepSeek corpus does. Invalid reading: that this predicts probe quality.
Section 3 shows the self-generated probes work at several layers while the weak
external ones almost never do, so repetition is not the variable that separates
working probes from failing ones. The weak external bar is blank because the R4
read was run on the E11 corpora only; the diverse corpus was never measured
either, and neither absence should be read as a zero.

</details>

**What this establishes.** Scaffold degeneracy is not the mechanism. The most
repetitive corpus in the comparison still yields probes that pass at several
layers (section 3), so hypothesis H-repetition from section 1 is not supported.
Repetition remains a real property of the self-generated corpus, and it stays
on the caveat list, but it does not explain the lineage gap.

**Live hypothesis (H-quality).** What drives probe function is generator
quality, not generator identity: a strong external writer should match or beat
the probed model's own stories. **Deciding read:** the detection grid in
section 3, which is E11's registered read R1.

**Open questions.** Phrase overlap is a shallow, surface-level measure of
diversity; it says nothing about semantic variety. The diverse corpus was never
put through this read, so we cannot state where the persona-and-setting grid
lands on this axis.

## 3. Whose stories give probes that detect at the most layers?

This is E11's registered detection read (registry name: read R1), the read the
experiment was designed around. For every lineage and every extracted layer, we
count how many of 12 test scenarios the probes place correctly, on each of the
two batteries, and ask whether the layer clears the registered bar on both.

The registered branch predictions were written before any story existed. The
*quality* branch said the strong external corpus would roughly match the
self-generated one and beat the weak external one. The *lineage* branch said
the self-generated corpus would beat both externals, which would have sat near
each other despite the quality gulf between their writers.

In [4]:
# section 3 is load-call-show: the per-layer dual-battery grid for all four
# lineages, with a slider over which battery the color shows
from emotion_vectors.lineage_report import detection_figure

fig_detection, detection_stats = detection_figure(evidence)
fig_detection.show()
print("\n".join(detection_stats["lines"]))

# LOAD-BEARING ANCHOR (keep this assert): the whole notebook's verdict is the
# ordering "strong external > self-generated > weak external" on passing-layer
# count, and section 7 quotes the three counts themselves. Both the ordering
# and the exact values are pinned here, so a re-score that preserved the
# ordering but moved a count would still halt the notebook rather than leave
# the verdict prose quietly stale.
n_passing = detection_stats["n_passing"]
expected_passing = {"fixed_deepseek": 9, "selfgen": 5, "weak_external": 1}
assert n_passing["fixed_deepseek"] > n_passing["selfgen"] > n_passing["weak_external"], (
    f"registered quality ordering violated: {n_passing}"
)
assert {key: n_passing[key] for key in expected_passing} == expected_passing, (
    f"passing-layer counts moved off the values section 7 quotes: {n_passing}"
)
print(
    "anchor check: fixed DeepSeek "
    f"({n_passing['fixed_deepseek']}) > self-generated ({n_passing['selfgen']})"
    f" > weak external ({n_passing['weak_external']}) passing layers,"
    " and each equals the value the verdict quotes: OK"
)

self-generated (stories by the probed model): 5 passing layers [33, 39, 42, 51, 54], best layer 39 (10/11)
weak external (stories by gemma-4-4B): 1 passing layers [42], best layer 42 (8/8)
fixed DeepSeek (stories by deepseek-v4-pro): 9 passing layers [6, 12, 33, 36, 39, 42, 45, 54, 57], best layer 39 (10/10)
diverse DeepSeek (deepseek-v4-pro, persona x setting grid): 7 passing layers [6, 9, 12, 36, 39, 42, 54], best layer 36 (11/9)
anchor check: fixed DeepSeek (9) > self-generated (5) > weak external (1) passing layers, and each equals the value the verdict quotes: OK


<details><summary><b>How to read this figure</b></summary>

Each row is one probe lineage, named by who wrote the stories those probes were
built from. Each column is one extracted layer of the probed model. One cell is
therefore one lineage at one layer, and its text is two numbers: how many of
the 12 scenarios that lineage placed correctly on the paper battery, then on
the held-out battery. A star marks a cell where both counts reach the
registered bar of 8.

The color encodes a single number per cell, and the slider under the figure
chooses which: the worse of the two batteries (the default, and the strict
view), the paper battery alone, or the held-out battery alone. The slider
changes what the color means, never the verdict, so the headline in the title
stays the same across steps while the subtitle restates which battery you are
looking at. The colorbar names both anchors: chance is 3 of 12 correct, since
ranking the target in the top 3 of 12 emotions by luck happens a quarter of the
time, and the pass bar is 8.

A good result here would be a row with many starred cells spread across layers.
A bad result would be a row hovering near the chance color with no stars at
all. The observed rows sit between those two: the exact passing-layer counts
are printed above and stated in the title, and the weak external row is the one
closest to the bad end.

Valid reading: which lineage's probes work, and at which depths. Invalid
reading: treating an unstarred cell with high counts as a near-miss worth
quoting, since the bar is a registered criterion and single cells are noisy;
section 4 shows how much a passing-layer count moves under reseeding.

</details>

**What this establishes.** The quality branch fired and the lineage branch did
not. Probes built from a strong external writer's stories pass at more layers
than probes built from the probed model's own stories, and both leave the weak
external corpus far behind, so what matters is that the stories are good, not
that the probed model wrote them. This is the observation behind claim C4,
which passed its falsification gate on 2026-07-23 (permutation null over the
full layer sweep, random-probe-set control, scenario bootstrap, an independent
dialogue-derived instrument, and a base-rate check; the scorecard is linked in
`TREE.md`). It also narrows the earlier corpus-dependence finding: probes are
corpus-dependent, but the dependence bites for *weak-generator* corpora.

**Live hypothesis (H-saturation).** The strong-generator advantage is an
efficiency effect, so a much smaller strong corpus should already reach the
same ceiling. **Deciding read:** the dose-response curves in section 4.

**Open questions.** One strong external generator was tested, so "quality" is
operationalised by a single point plus the weak comparison. The band edges are
the honest scope limit: the falsification gate found the core layers stable
under scenario bootstrap but the outermost passing layers fragile.

## 4. How many stories do probes need, and does prompt diversity help?

E11 answered the *quality* half of the question. This is E12, which answers
the *how much* and *how varied* half. Two arms are subsampled to a ladder of
corpus sizes, five random seeds at each size, and each subsample is scored
through the identical detection read from section 3.

The registered branch predictions, again written before the diverse corpus
existed: the *saturation* branch said both arms would flatten by roughly 128 to
256 stories per emotion and that matching sizes would show no diversity gain;
the *growth* branch said the diverse arm would beat E11's passing-layer count
or would win at matched size.

In [5]:
# section 4 is load-call-show: passing layers and probe-direction convergence
# against corpus size, with the E11 results drawn as reference lines
from emotion_vectors.lineage_report import dose_response_figure

fig_dose, dose_stats = dose_response_figure(evidence)
fig_dose.show()
print("\n".join(dose_stats["lines"]))

fixed-prompt DeepSeek stories, seed-mean passing layers by n: n=8: 3.8, n=16: 5.0, n=32: 6.2, n=64: 8.6, n=128: 7.8, n=255: 9.0
diverse-prompt DeepSeek stories, seed-mean passing layers by n: n=8: 0.8, n=16: 2.4, n=32: 2.4, n=64: 4.0, n=128: 6.4, n=256: 6.2, n=512: 6.8, n=1018: 7.0
matched-n detection: diverse at n=256 6.2 layers vs fixed at its full corpus n=255 9.0 layers; diverse at full n=1018 reaches 7.0, ceiling 9


<details><summary><b>How to read this figure</b></summary>

Both panels share a horizontal axis: stories per emotion, on a log scale, so
each step to the right is a doubling. One faint marker is one seeded subsample;
the solid line joins the average over the five seeds at each size. Blue is the
fixed-prompt DeepSeek corpus, red is the diverse-prompt one; the legend names
both roles. The largest point in each arm is that arm's whole corpus, where
subsampling does nothing, and its size is the smallest per-emotion count
printed back in section 1.

The left panel's vertical axis counts how many of the 20 extracted layers pass
the dual-battery bar. Its three reference lines are the grading scale: the
fixed corpus's full-size result at the top as the ceiling to beat, the
self-generated n=256 result as the comparator from E11, and zero at the bottom
as outright failure. The right panel's vertical axis is the mean cosine between
a subsample's probe directions and the self-generated probe directions at layer
33, with the full fixed corpus as the upper anchor and zero (unrelated
directions) as the failure anchor. There is no layer slider on the right panel
because the registered geometry read exists at layer 33 only.

A good result for "more data helps" would be a curve still climbing at the
right edge. A bad one would be a flat curve from early on. The left panel's
blue curve is the flat kind past its knee, and the right panel's curves are
still climbing, which is the interesting mismatch: probe *function* saturates
before probe *geometry* stops moving.

Valid reading: how much corpus a detection probe needs, and how the two arms
compare at equal size. Invalid reading: treating the vertical wobble between
adjacent sizes as signal. Five seeds at one size span several layers, visible
as the vertical spread of faint markers, so read the level of a curve rather
than a single step. Also invalid: comparing the right panel's largest-n values
against section 5's matrix as if they were the same measurement. The
dose-response points come from subsampled per-story residuals, section 5 uses
the full-corpus bundles, and they differ in the third decimal.

</details>

**What this establishes.** Detection saturates early, and prompt diversity is a
cost rather than a benefit for this read. The fixed arm is near its ceiling by
64 stories per emotion, and 64 strong-generator stories already beat the 256
self-generated ones from E11 (all four numbers are printed above and stated in
the figure title). The diverse arm sits below the fixed arm at every matched
size and never reaches the fixed ceiling even at its full corpus. The
registered growth branch did not fire; the diversity read fired in reverse.

The mechanism we propose: the persona-and-setting grid injects non-emotional
variance into every story, so each emotion's mean is estimated from noisier
samples and needs more of them for the same precision.

**Live hypothesis (H-diversity-pays-elsewhere).** The variance the grid adds is
not junk; it is coverage, which should help a read that needs finer resolution
than a top-3 ranking. **Deciding read:** the preference correlation in section
6, including its matched-size check.

**Open questions.** The diverse arm changed the prompt, not only the diversity.
The grid is identical across all 12 emotions, so setting content is a shared
component that the 12-pool centering removes by construction, but second-order
interactions between a setting and an emotion are not controlled. A cleaner
test would hold the prompt fixed and vary only the sampling temperature.

## 5. Do the probe directions themselves converge?

Sections 3 and 4 measured what probes *do*. This is E11's registered geometry
read (registry name: read R2), which measures what they *are*: how close the
directions of two probe sets are, and whether the two sets organise the 12
emotions the same way relative to each other.

The read is defined at layer 33 only. That is the geometry peak identified in
notebook 02, and it was registered as a single-layer read before scoring, so
the frozen evidence has no per-layer version of this comparison to scrub
through.

In [6]:
# section 5 is load-call-show: the symmetric lineage-by-lineage agreement matrix
# at the registered geometry layer
from emotion_vectors.lineage_report import geometry_figure

fig_geometry, geometry_stats = geometry_figure(evidence)
fig_geometry.show()
print("\n".join(geometry_stats["lines"]))

self-generated (stories by the probed model) vs weak external (stories by gemma-4-4B): contrast cos 0.225, RSA 0.275
self-generated (stories by the probed model) vs fixed DeepSeek (stories by deepseek-v4-pro): contrast cos 0.568, RSA 0.600
weak external (stories by gemma-4-4B) vs fixed DeepSeek (stories by deepseek-v4-pro): contrast cos 0.284, RSA 0.441
self-generated (stories by the probed model) vs diverse DeepSeek (deepseek-v4-pro, persona x setting grid): contrast cos 0.459, RSA 0.700
fixed DeepSeek (stories by deepseek-v4-pro) vs diverse DeepSeek (deepseek-v4-pro, persona x setting grid): contrast cos 0.807, RSA 0.872


<details><summary><b>How to read this figure</b></summary>

Both axes list the same four probe lineages, named by who wrote the stories the
probes were built from, so one cell is one pair of probe sets. The matrix is
symmetric: the cell in row A, column B says the same thing as row B, column A.

Each off-diagonal cell holds two numbers. The top one, which is also the cell's
color, is the mean over the 12 emotions of the cosine between the two sets'
directions for that emotion: it asks whether the directions point the same way.
The bottom one is the RSA, the correlation between the two sets' 12x12
emotion-similarity matrices: it asks whether the two sets place emotions in the
same arrangement relative to one another, which can hold even when individual
directions have rotated. The colorbar names the anchors: 1 is the same
direction, 0 is unrelated, and -1 is opposed. The diagonal is 1 by definition,
a probe set compared against itself, and is the strength anchor; 0 is the
failure anchor. Two cells say "pair not measured": they are the same unmeasured pair
shown twice, and a blank is not a zero.

A good result for "a better writer converges toward the probed model's own
geometry" would be a self-generated row where cosine rises with generator
quality. A bad one would be a flat row, every external corpus equally far away.
The observed row is the good-shaped one but only partly: the exact values are
printed above and in the title.

Valid reading: the strong external writer recovers substantially more of the
probed model's own directions than the weak one does. Invalid reading: treating
a cosine of about a half as "the same probe". Half is a long way from 1, which
is precisely why the caveat in section 7 calls the emotion direction a
convention-qualified object rather than a single thing.

</details>

**What this establishes.** The mechanism behind section 3 is at least partly
geometric convergence: a stronger writer produces stories that push the probed
model into more nearly the same directions its own stories do. The diverse arm
behaves distinctively, keeping the *shape* of emotion space much better than it
keeps the individual directions, which is consistent with the section 4 reading
that its extra variance rotates each emotion's estimate without disturbing
their arrangement.

**Live hypothesis (H-partial-convergence).** Convergence is bounded: no
external corpus, however strong the writer, will drive the cosine near 1,
because part of the direction encodes the writer's own style. **Deciding read:**
a second strong external generator, unrun. If a different strong writer lands
at a similar cosine, the ceiling is a property of externality; if it lands
higher, the ceiling was just deepseek-v4-pro's style.

**Open questions.** The read exists at one layer, so we cannot say whether
convergence is stronger or weaker at the layers where detection actually
passes. Of the six distinct lineage pairs, five are printed above and one, weak
external against diverse DeepSeek, was never measured; it is the pair drawn
twice in the symmetric matrix.

## 6. Where does corpus quality still pay?

Detection is a coarse read: it only asks whether the right emotion made the top
3. This is E11's registered preference read (registry name: read R3), which is
finer grained. For each of 64 activities the model was asked to compare
pairwise, an Elo rating summarises how much it says it prefers that activity.
The read then asks how strongly a probe's centered cosine tracks that rating.

Section 4 left a hypothesis open: that the diverse corpus's extra variance is
coverage, and coverage should pay on a read that needs more resolution than a
top-3 ranking. This is that read.

In [7]:
# section 6 is load-call-show: the best probe-to-Elo correlation per lineage,
# with the exploratory matched-size check overlaid on the diverse bar
from emotion_vectors.lineage_report import preference_figure

fig_preference, preference_stats = preference_figure(evidence)
fig_preference.show()
print("\n".join(preference_stats["lines"]))

self-generated (stories by the probed model): max |r| = 0.616 (layer 33, calm)
weak external (stories by gemma-4-4B): max |r| = 0.593 (layer 30, calm)
fixed DeepSeek (stories by deepseek-v4-pro): max |r| = 0.706 (layer 24, angry)
diverse DeepSeek (deepseek-v4-pro, persona x setting grid): max |r| = 0.772 (layer 24, angry)
matched-n check, diverse subsampled to n=256: mean |r| 0.760 (5 seeds, 0.746 to 0.772)


<details><summary><b>How to read this figure</b></summary>

One bar is one probe lineage, named on the horizontal axis by who wrote its
stories. Its height is the largest absolute Pearson correlation, over 12
emotions and 4 measured layers, between that emotion's probe cosine and the
preference Elo across the 64 activities. The label on each bar names which
layer and which emotion won that maximum. The vertical axis runs from 0, no
linear relation at all, toward 1, a perfect one.

The two grading anchors are labelled on the right: the dotted line is the
weakest generator's level, the comparator any better corpus should clear, and 0
is the failure anchor. The black diamond on the diverse bar is a separate,
exploratory measurement: the diverse corpus subsampled down to 256 stories per
emotion, five seeds, mean and range. It exists to ask whether the diverse
corpus's advantage is really diversity or just its four-times-larger size. Its
whisker is short because the seeds agree closely, not because it is missing.

A good result for "quality and diversity pay here" would be an ordering that
climbs with generator quality and survives the size match. A bad one would be
four bars at the same level, or a diamond that falls back to the fixed
corpus's bar. Observed: the ordering climbs and the diamond stays high, with
the exact values printed above.

Valid reading: the levels, and the fact that the ordering holds at matched
size. Invalid reading: any small gap between adjacent bars. Each bar is a
maximum over 48 candidate combinations, which inflates it, and this read is a
three-point trend reported as exploratory support, not a gated claim.

</details>

**What this establishes.** The section 4 hypothesis H-diversity-pays-elsewhere
is supported. Coarse detection wants a clean, low-variance corpus; the
fine-grained behavioural correlate wants coverage. The matched-size check says
the diverse corpus's advantage is not simply that it is bigger, since
subsampling it to the fixed corpus's size leaves it above the fixed corpus's
level. The practical consequence is that "which corpus is best" has no
corpus-independent answer: it depends on what the probe will be asked to do.

**Live hypothesis (H-resolution).** The split between the two reads is about
resolution, not about emotion versus preference, so any fine-grained read
(intensity ratings, graded similarity judgements) should favour the diverse
corpus while any coarse ranking read should favour the fixed one. **Deciding
read:** run a graded-intensity read on both arms, unrun.

**Open questions.** This is a maximum statistic over 48 combinations with no
permutation null attached, so its absolute level is not trustworthy even though
the ordering is stable across the matched-size seeds. The winning emotion
differs across lineages, which is not explained.

## 7. What we are saying, exactly

The cell below recomputes, from the frozen evidence files, every number the
verdict cites. Read the verdict against that printout line by line.

In [8]:
# section 7 is load-call-show: the verdict's number-check record
from emotion_vectors.lineage_report import verdict_stats

verdict = verdict_stats(evidence)
print("\n".join(verdict["lines"]))

# LOAD-BEARING ANCHORS (keep these asserts): the three claims the verdict below
# states in words. Each is pinned to the evidence so a re-score cannot leave the
# prose describing a result that has changed underneath it.
assert verdict["n_passing"]["fixed_deepseek"] > verdict["n_passing"]["selfgen"], (
    "verdict item 1 (quality beats identity) no longer holds"
)
matched = verdict["matched_detection"]
assert matched["diverse_n256"] < matched["fixed_full"], (
    "verdict item 3 (diversity is a detection tax at matched size) no longer holds"
)
assert verdict["pref_matched_mean"] > verdict["pref_abs_r"]["fixed_deepseek"], (
    "verdict item 4 (diversity pays on the preference read at matched size) no longer holds"
)
print("anchor check: all three verdict orderings hold on the current evidence: OK")

R1 self-generated (stories by the probed model): 5 passing layers [33, 39, 42, 51, 54]
R1 weak external (stories by gemma-4-4B): 1 passing layers [42]
R1 fixed DeepSeek (stories by deepseek-v4-pro): 9 passing layers [6, 12, 33, 36, 39, 42, 45, 54, 57]
R1 diverse DeepSeek (deepseek-v4-pro, persona x setting grid): 7 passing layers [6, 9, 12, 36, 39, 42, 54]
E12 fixed arm, seed-mean passing layers: n=8: 3.8, n=16: 5.0, n=32: 6.2, n=64: 8.6, n=128: 7.8, n=255: 9.0
E12 diverse arm, seed-mean passing layers: n=8: 0.8, n=16: 2.4, n=32: 2.4, n=64: 4.0, n=128: 6.4, n=256: 6.2, n=512: 6.8, n=1018: 7.0
matched-n detection: diverse n=256 6.2 vs fixed full corpus n=255 9.0; diverse full n=1018 7.0
preference max |r|: self-generated (stories by the probed model) 0.616, weak external (stories by gemma-4-4B) 0.593, fixed DeepSeek (stories by deepseek-v4-pro) 0.706, diverse DeepSeek (deepseek-v4-pro, persona x setting grid) 0.772
preference matched-n, diverse at n=256: 0.760 (seeds 0.746 to 0.772)
anc

Every number in this list is printed by the cell above.

1. **Generator quality, not generator identity, drives probe function** (E11,
   with claim C4 through its falsification gate). Probes built from
   fixed-DeepSeek stories pass at 9 layers, against 5 for the probed model's own
   stories and 1 for the weak external corpus.
2. **Detection probes saturate early** (E12). The fixed arm is at 8.6 of its
   9-layer ceiling by 64 stories per emotion, and those 64 strong-generator
   stories already beat the 256 self-generated ones.
3. **Prompt diversity is a COST for detection at matched size** (E12). The
   diverse arm averages 6.2 passing layers at 256 stories per emotion against
   the fixed arm's 9.0 at its full 255, and even the full diverse corpus at
   1018 reaches only 7.0, below the fixed ceiling of 9. The registered growth
   branch did not fire; the diversity read fired in reverse.
4. **The same diversity HELPS the fine-grained preference read, at matched
   size**. Subsampled to 256 stories per emotion, the diverse corpus scores
   0.760 against the fixed corpus's 0.706, and the full ordering runs 0.616 for
   self-generated, 0.706 for fixed DeepSeek, 0.772 for diverse DeepSeek. This is
   exploratory support, not a gated claim.
5. **Practical recipe.** For detection probes, roughly 64 to 256 fixed-prompt
   stories per emotion from any strong generator. Add prompt diversity only when
   the downstream read is fine grained, and expect to pay a detection tax for
   it.

### The caveat that qualifies all of it

"The emotion direction" is not one object. It is qualified by three
conventions, and changing any of them changes the vector. The first is corpus
lineage, which is this whole notebook. The second is readout centering, settled
in experiment E9, whose centered cosine is what every read above uses. The
third is the extraction format: whether the stories are fed to the model as
plain text or wrapped in its chat template. Every number in this notebook comes
from raw-text extraction. The audit below (registry name: E4b) re-extracted the
same self-generated corpus through the chat template and compared.

In [9]:
# the caveat exhibit is load-call-show too: raw-text against chat-template
# extraction, layer by layer, from the E4b audit
from emotion_vectors.lineage_report import extraction_format_figure

fig_format, format_stats = extraction_format_figure(evidence)
fig_format.show()
print("\n".join(format_stats["lines"]))

raw-vs-chat extraction agreement per layer (contrast mean / worst emotion / raw mean):
  layer 24: 0.75 / 0.56 (nervous) / 1.00
  layer 30: 0.57 / 0.25 (angry) / 0.98
  layer 33: 0.58 / 0.18 (angry) / 0.99
  layer 36: 0.52 / 0.14 (angry) / 0.99
  layer 39: 0.60 / 0.30 (angry) / 0.98
  layer 42: 0.69 / 0.34 (angry) / 0.99
  layer 57: 0.71 / 0.53 (angry) / 0.92
contrast means 0.52 to 0.75; worst per-emotion agreement 0.14 (angry, layer 36); uncentered means 0.92 to 1.00


<details><summary><b>How to read this figure</b></summary>

The horizontal axis is the layer, ticked only at the layers the audit actually
covered, so the wide gap before the last tick is real absence, not a flat
stretch. The vertical axis is the cosine between the raw-text and the
chat-template version of the *same* probe, built from the *same* stories. One
point is one layer.

Three series, each named in the legend by its role. The gray dashed line is the
uncentered mean vectors, before the 12-pool subtraction. The thick blue line is
the contrast directions averaged over the 12 emotions, which is the object
every figure in this notebook uses. The red dotted line is the single worst
emotion at each layer, hover to see which. The horizontal lines are the grading
scale: 1.0 would mean the format changed nothing, 0.9 is a reading aid for
"close enough to treat the two formats as one instrument", and 0 at the bottom
is the failure anchor. E4b registered no pass bar on this read, so 0.9 is our
convention rather than a registered criterion.

A good result for "the format does not matter" would be all three series above
the 0.9 line. A bad one would be the working series near 0. Observed: the
uncentered series is in the good region while the contrast series, the one that
matters, sits roughly midway, with the worst single emotion dropping much
further. The exact ranges are in the title and the printout.

Valid reading: the 12-pool centering, which is what makes these directions
emotion-specific, is also what makes them format-specific. Invalid reading:
concluding the vectors are meaningless. They are internally consistent within a
convention, which is why every comparison in this notebook holds the convention
fixed; what fails is transporting a direction across conventions.

</details>

**The rest of the caveat list.**

- Only one strong external generator was tested. "Quality" is operationalised
  by that single point plus the weak gemma-4-4B corpus, so the quality axis has
  two levels, not a scale.
- The diverse arm changed the prompt, not only the diversity. Persona and
  setting content are shared across all 12 emotions and removed by the 12-pool
  centering by construction, but second-order interactions between a setting and
  an emotion are not controlled.
- The preference ordering in section 6 is a three-point trend on a maximum
  statistic with no permutation null. It is reported as exploratory support and
  is not a gated claim.
- The phrase-repetition read in section 2 is exploratory and covers two of the
  four lineages.
- The detection band edges are fragile. The falsification gate for claim C4
  found the core passing layers stable under scenario bootstrap while the
  outermost ones were not, and that is the honest scope limit on the
  passing-layer counts quoted throughout.

**Open questions this notebook does not answer.** Whether a second strong
generator lands at the same geometric ceiling as deepseek-v4-pro. Whether the
detection-versus-preference split is really about read resolution, as
hypothesis H-resolution proposes. Whether a diversity manipulation that does
not change the prompt (sampling temperature, for instance) still costs
detection accuracy. Each is a straightforward follow-up experiment, and none is
run.